# M0 milestone notebook: Cello UCF gate -> Pacti polyhedral contracts

This notebook implements the M0 slice for your fork:

1. Load a Cello-like UCF JSON gate definition (`ymin`, `ymax`, `K`, `n`).
2. Work in **linear REU**.
3. Build **4 piecewise linear segments** with **sound output bounds** per segment.
4. Emit one `PolyhedralIoContract` per segment.
5. Validate on a dense grid that the Hill response stays inside each segment contract.

> Note: The bounds here are mathematically sound because each segment uses monotonic-range bounds for a NOT Hill gate over an interval.

In [ ]:
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List

import numpy as np

from pacti.contracts import PolyhedralIoContract

## 1) Load one gate from a UCF-style JSON source

In [ ]:
# Minimal Cello-like gate record (edit this to point at your real UCF extract if desired)
example_gate = {
    "name": "P3_PhlF",
    "ymin": 0.08,
    "ymax": 320.0,
    "K": 14.0,
    "n": 2.4,
}

# Optional: write a tiny UCF-like file so the loader path is explicit
ucf_path = Path("examples/cello_ucf_gate_minimal.json")
ucf_path.parent.mkdir(parents=True, exist_ok=True)
ucf_path.write_text(json.dumps({"gates": [example_gate]}, indent=2))
print(f"Wrote {ucf_path}")


def _find_gate_records(obj: Any) -> List[Dict[str, Any]]:
    """Recursively find dicts that look like gate transfer parameter records."""
    records = []
    if isinstance(obj, dict):
        keys = set(obj.keys())
        if {"name", "ymin", "ymax", "K", "n"}.issubset(keys):
            records.append(obj)
        for v in obj.values():
            records.extend(_find_gate_records(v))
    elif isinstance(obj, list):
        for item in obj:
            records.extend(_find_gate_records(item))
    return records


def load_gate_from_ucf_json(path: Path, gate_name: str) -> Dict[str, float]:
    data = json.loads(path.read_text())
    candidates = _find_gate_records(data)
    for rec in candidates:
        if rec["name"] == gate_name:
            return {
                "name": rec["name"],
                "ymin": float(rec["ymin"]),
                "ymax": float(rec["ymax"]),
                "K": float(rec["K"]),
                "n": float(rec["n"]),
            }
    raise ValueError(f"Gate {gate_name!r} not found in {path}")


gate = load_gate_from_ucf_json(ucf_path, "P3_PhlF")
gate

## 2) Define Hill response and piecewise contract extraction (4 segments)

In [ ]:
@dataclass
class SegmentContract:
    x_lo: float
    x_hi: float
    y_lo: float
    y_hi: float
    contract: PolyhedralIoContract


def hill_not_response(x: np.ndarray, *, ymin: float, ymax: float, K: float, n: float) -> np.ndarray:
    """Repressive Hill response in linear REU."""
    return ymin + (ymax - ymin) / (1.0 + (x / K) ** n)


def build_piecewise_sound_contracts(
    gate: Dict[str, float],
    x_var: str,
    y_var: str,
    x_min: float,
    x_max: float,
    num_segments: int = 4,
) -> List[SegmentContract]:
    """Build sound segment contracts using monotonic interval output bounds.

    For a monotone decreasing Hill NOT gate on [a,b], exact output range is [f(b), f(a)].
    We encode that as linear inequalities y>=f(b), y<=f(a), plus input interval assumptions.
    """
    edges = np.linspace(x_min, x_max, num_segments + 1)
    segs: List[SegmentContract] = []

    for i in range(num_segments):
        a = float(edges[i])
        b = float(edges[i + 1])
        fa = float(hill_not_response(np.array([a]), **{k: gate[k] for k in ["ymin", "ymax", "K", "n"]})[0])
        fb = float(hill_not_response(np.array([b]), **{k: gate[k] for k in ["ymin", "ymax", "K", "n"]})[0])

        y_lo = min(fa, fb)
        y_hi = max(fa, fb)

        assumptions = [
            f"{x_var} <= {b}",
            f"-{x_var} <= {-a}",
        ]
        guarantees = [
            f"{y_var} <= {y_hi}",
            f"-{y_var} <= {-y_lo}",
        ]

        c = PolyhedralIoContract.from_strings(
            input_vars=[x_var],
            output_vars=[y_var],
            assumptions=assumptions,
            guarantees=guarantees,
        )
        segs.append(SegmentContract(a, b, y_lo, y_hi, c))

    return segs


x_min, x_max = 1e-3, 1e3
segments = build_piecewise_sound_contracts(
    gate=gate,
    x_var="x_in",
    y_var="x_out",
    x_min=x_min,
    x_max=x_max,
    num_segments=4,
)

for idx, s in enumerate(segments, start=1):
    print(f"Segment {idx}: x in [{s.x_lo:.4g}, {s.x_hi:.4g}] => y in [{s.y_lo:.4g}, {s.y_hi:.4g}]")

## 3) Inspect generated Pacti contracts

In [ ]:
for idx, s in enumerate(segments, start=1):
    print(f"\n=== Contract segment {idx} ===")
    print(s.contract)

## 4) Dense-grid validation (enclosure check)

In [ ]:
def validate_enclosure(gate: Dict[str, float], segments: List[SegmentContract], n_per_segment: int = 5000) -> None:
    params = {k: gate[k] for k in ["ymin", "ymax", "K", "n"]}
    for idx, s in enumerate(segments, start=1):
        xs = np.linspace(s.x_lo, s.x_hi, n_per_segment)
        ys = hill_not_response(xs, **params)
        ok = np.all((ys >= s.y_lo - 1e-12) & (ys <= s.y_hi + 1e-12))
        if not ok:
            raise AssertionError(f"Segment {idx} failed enclosure")
    print(f"All {len(segments)} segments enclose the Hill response on dense samples.")


validate_enclosure(gate, segments)

## 5) Notes for next step (toward your "replicate Cello example" goal)

- This notebook gives a sound contract abstraction for one characterized gate in linear REU.
- To replicate one full Cello example design, next add:
  1. real UCF parsing for your target organism/library,
  2. multiple gates + wiring map from a chosen Cello example netlist,
  3. contract composition checks through that topology.